# Cold versus learned execution on a Fabric semantic model

This notebook runs the same complex analytical question against **ARR Model SF (79)** twice:

1. Cold execution with a directly bound `SemanticModel`.
2. Learned execution after `RLM.learn()` profiles the model and saves a portable package to OneLake.

The comparison records correctness, turns, tokens, model time, worker time, wall time, and knowledge provenance. The current branch does not execute registered learned operations, so it does **not** guarantee fewer turns or lower latency yet. Its current improvement is source preflight, portable rebinding, drift rejection, and auditable provenance.

## 1. Install directly from the feature branch

The repository is public, so no wheel download or attached Lakehouse is required.

In [ ]:
%pip install -q --upgrade "git+https://github.com/pawarbi/fabric-rlm-core.git@feature/knowledge-package-rebinding-integrity"

## 2. Configure the model and package location

Both resources are addressed explicitly by ID. The notebook has no default Lakehouse attachment. The package is saved to an ABFSS location through the public API's default OneLake transport.

In [ ]:
import json
import time

import fabric_rlm
from fabric_rlm import FabricLM, RLM, SemanticModel, load_knowledge

WORKSPACE_ID = "2680c303-be42-4d4a-b230-281d2cedf17b"
MODEL_ID = "f76244f0-6352-4947-bbaf-98ad3f76f96c"
# Existing Lakehouse item used only as the OneLake Files destination.
# It does not need to be attached to this notebook.
ONELAKE_ITEM_ID = "54511b33-e765-469b-8d04-84df03d623bf"
KNOWLEDGE_STORE = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
    f"{ONELAKE_ITEM_ID}/Files/knowledge-demo/arr-model-sf-79.json"
)

model = SemanticModel(MODEL_ID, workspace=WORKSPACE_ID)
print("fabric-rlm", fabric_rlm.__version__)
print("semantic model", model)
print("knowledge store", KNOWLEDGE_STORE)

## 3. Define one grounded complex question

The model is quarterly and requires `ARR Data[IS_QUARTER] = 1`. ARR must be evaluated for individual quarters rather than summed across quarters. Customer geography comes from the `Owner` table.

In [ ]:
QUESTION = """
For the most recently completed quarter, 2026/Q2, ARR fell versus the prior
quarter even though it remained up year over year. Determine which customer
regions and product-line groups drove the quarter-over-quarter decline, and
which region x product-line combinations contributed most.

Use the semantic model's defined measures and relationships. Always filter
ARR Data[IS_QUARTER] to 1. Evaluate ARR separately for 2026/Q2, 2026/Q1,
and 2025/Q2; never sum ARR across quarters. Use Owner[Owner Region] for
customer geography. Include 2026/Q2 quantity, absolute and percentage QoQ
and YoY changes, the largest negative product-line drivers, the largest
negative regional drivers, and the largest negative region x product-line
combinations. Reconcile the driver analysis to the overall change.
""".strip()

OUTPUTS = {
    "arr_2026_q2": float,
    "arr_2026_q1": float,
    "arr_2025_q2": float,
    "qoq_change": float,
    "qoq_percent": float,
    "yoy_change": float,
    "yoy_percent": float,
    "quantity_2026_q2": int,
    "top_product_line_driver": str,
    "top_region_driver": str,
    "top_region_product_driver": str,
    "analysis": str,
}

EXPECTED = {
    "arr_2026_q2": 237576169.6,
    "arr_2026_q1": 249915027.1,
    "arr_2025_q2": 234669398.8,
    "qoq_change": -12338857.6,
    "qoq_percent": -4.93722116,
    "yoy_change": 2906770.7,
    "yoy_percent": 1.238666275,
    "quantity_2026_q2": 107507,
}

print(QUESTION)

## 4. Run the cold baseline

In [ ]:
cold_lm = FabricLM("gpt-5.1", reasoning_effort="medium")
cold_started = time.perf_counter()
cold_result = RLM.task(
    QUESTION,
    inputs={"arr_model": model},
    outputs=OUTPUTS,
    lm=cold_lm,
    max_turns=10,
).run()
cold_wall_seconds = time.perf_counter() - cold_started

print(json.dumps(cold_result.payload, indent=2, default=str))
print(cold_result.report())

## 5. Learn and save to ABFSS

No `transport=` argument is needed. An ABFSS store automatically uses the built-in OneLake REST transport.

`RLM.learn()` performs bounded metadata profiling and a stability check. This one-time package creation cost is reported separately from both task runs.

In [ ]:
learn_started = time.perf_counter()
knowledge = RLM.learn(
    sources={"arr_model": model},
    store=KNOWLEDGE_STORE,
    overwrite=True,
)
learn_wall_seconds = time.perf_counter() - learn_started

source_profile = knowledge.package.sources[0]
print({
    "package_id": knowledge.package.package_id,
    "package_fingerprint": knowledge.package.fingerprint,
    "source_family": source_profile.family,
    "source_status": source_profile.status,
    "learn_wall_seconds": round(learn_wall_seconds, 3),
    "registered_operations": len(knowledge.package.operations),
})

## 6. Rebind from the saved package

This simulates a fresh notebook or session. The portable package is loaded from ABFSS and rebound to the currently authorized semantic-model handle.

In [ ]:
loaded_knowledge = load_knowledge(
    KNOWLEDGE_STORE,
    sources={
        "arr_model": SemanticModel(MODEL_ID, workspace=WORKSPACE_ID),
    },
)

assert loaded_knowledge.package.fingerprint == knowledge.package.fingerprint
print("rebound package", loaded_knowledge.package.fingerprint)

## 7. Ask the same question with learned knowledge

In [ ]:
learned_lm = FabricLM("gpt-5.1", reasoning_effort="medium")
learned_started = time.perf_counter()
learned_result = RLM.task(
    QUESTION,
    knowledge=loaded_knowledge,
    outputs=OUTPUTS,
    lm=learned_lm,
    max_turns=10,
).run()
learned_wall_seconds = time.perf_counter() - learned_started

print(json.dumps(learned_result.payload, indent=2, default=str))
print(learned_result.report())

## 8. Compare measured outcomes

The expected values below were independently queried from the model. Numeric correctness uses a small tolerance because providers may round displayed values.

In [ ]:
def numeric_correct(payload):
    if not payload:
        return False
    for field, expected in EXPECTED.items():
        actual = payload.get(field)
        if actual is None:
            return False
        tolerance = 0.05 if field.endswith("_percent") else max(1.0, abs(expected) * 0.0001)
        if abs(float(actual) - expected) > tolerance:
            return False
    return True

def run_metrics(label, result, wall_seconds):
    metadata = result.trajectory.metadata
    return {
        "run": label,
        "submitted": result.submitted,
        "numeric_correct": numeric_correct(result.payload),
        "turns": result.n_turns,
        "prompt_tokens": result.total_prompt_tokens,
        "completion_tokens": result.total_completion_tokens,
        "lm_seconds": result.total_lm_seconds,
        "worker_seconds": result.total_worker_seconds,
        "wall_seconds": round(wall_seconds, 3),
        "knowledge_fingerprint": metadata.get("knowledge_fingerprint"),
        "knowledge_mode": metadata.get("knowledge_mode"),
    }

comparison = [
    run_metrics("cold", cold_result, cold_wall_seconds),
    run_metrics("learned", learned_result, learned_wall_seconds),
]
print(json.dumps(comparison, indent=2, default=str))

cold = comparison[0]
learned = comparison[1]
execution_improved = (
    cold["numeric_correct"]
    and
    learned["numeric_correct"]
    and (
        learned["turns"] < cold["turns"]
        or (
            learned["prompt_tokens"] is not None
            and cold["prompt_tokens"] is not None
            and learned["prompt_tokens"] < cold["prompt_tokens"]
        )
        or learned["wall_seconds"] < cold["wall_seconds"]
    )
)

print({
    "measured_execution_improvement": execution_improved,
    "governance_metadata_present": bool(
        not cold["knowledge_fingerprint"]
        and
        learned["knowledge_fingerprint"]
        and learned["knowledge_mode"]
    ),
    "knowledge_mode": learned["knowledge_mode"],
})

## How to interpret the result

- `numeric_correct` shows whether each run reproduced the independently established model totals.
- `measured_execution_improvement` is true only when both runs are correct and the learned run actually reduces turns, prompt tokens, or wall time in this run.
- `governance_metadata_present` contrasts the learned run's source-bound fingerprint and explicit execution mode with the cold run.
- `learn_wall_seconds` is the separately reported one-time profiling and package-publication cost. Loading and task-time preflight also recheck the live model before reuse, so the learned path currently trades extra metadata calls for fail-closed validation.
- `fallback_no_registered_operations` is expected on this branch. It means knowledge validated and rebound the source, but the task still used normal RLM discovery. Therefore an execution-efficiency improvement is possible through model variance, but it is not a product guarantee until typed registered operations are implemented.